# Implement Cosine Similarity

**Easy** &nbsp;·&nbsp; TensorTonic &nbsp;·&nbsp; `Linear Algebra`

Implement a function to compute the cosine similarity between two vectors `a` and
`b`. Cosine similarity measures the **angle** between two vectors in a
high-dimensional space — not their length.

$$\text{cosine\_similarity}(a, b) = \frac{a \cdot b}{\|a\| \, \|b\|}$$

where:

- $a \cdot b$ is the dot product
- $\|a\|$ and $\|b\|$ are the Euclidean norms of `a` and `b`

---

**Example 1:**

```
Input:  a = [1, 2, 3], b = [2, 4, 6]
Output: 1.0    (perfect alignment - b is just a scaled up)
```

**Example 2:**

```
Input:  a = [1, 0], b = [0, 1]
Output: 0.0    (orthogonal)
```

---

**Hint 1:** use `np.dot()` for the dot product and `np.linalg.norm()` for the norms.

**Hint 2:** handle zero vectors gracefully by checking if either norm is 0 and
returning 0.0 in that case.

**Requirements:**

- input: 1-D NumPy arrays of equal length
- output: scalar float
- must be fully vectorized (no loops)
- handle zero vectors gracefully (return 0.0 if either norm is 0)

**Constraints:**

- `len(a) == len(b) <= 10^5`
- use only NumPy

---

You have already written both halves of this: the dot product on top, the
Euclidean norm on the bottom.

### The part that decides whether you pass

Hint 2 is not a nicety, it is the graded case. A zero vector has no direction, so
the angle to it is undefined, and the formula divides by zero.

The trap is *where* you check. Two orderings look equally sensible:

- compute the result, then if it came out `nan`, return `0.0`
- compute the two norms, and if either is `0`, return `0.0` before dividing

Only one of them is right. Work out why — think about what NumPy actually does at
the moment of the division, and whether "check afterwards" leaves a warning or a
`nan` behind that you never see in a test but will see in production.

### A second version worth writing

Divide each vector by its own norm first, then take a plain dot product of the
two unit vectors. Same answer.

That looks like pointless extra work for a single pair — and it is. Now imagine
comparing one query against a million stored documents. Which parts of that
calculation could you have done **once, in advance**? That is why every vector
database stores its vectors pre-normalised.

### A third, if you want the shape you will actually use

`cosine_similarity_matrix(a, matrix)` — one vector against every **row** of a 2-D
array, returning one similarity per row, with no Python loop over the rows. Two
things to sort out: how to get a norm per row rather than one for the whole array
(look at the `axis` parameter), and how to do the division when *some* rows are
zero vectors and must come out as `0.0` while the rest divide normally. NumPy has
a `where=` parameter for exactly that.

In [18]:
import numpy as np


class Solution:
    def dot_product(self, x, y) -> float:
        if len(x) != len(y):
            raise ValueError("Vectors must have the same length")
        return float(np.dot(x, y))

    def norm(self,a):
        return np.sqrt(np.dot(a, a))

    def cosine_similarity(self, a, b) -> float:
        a_norm = self.norm(a)
        b_norm = self.norm(b)
        if a_norm == 0 or b_norm == 0:
            return 0.0
        return self.dot_product(a,b)/(a_norm*b_norm)

    # optional: normalise both to unit length first, then dot
    def cosine_similarity_unit(self, a, b) -> float:
        a_norm = self.norm(a)
        b_norm = self.norm(b)
        if a_norm == 0 or b_norm == 0:
            return 0.0
        a_hat = a / a_norm
        b_hat = b / b_norm
        return self.dot_product(a_hat, b_hat)
    # optional: one vector vs every row of a matrix -> array of similarities
    def cosine_similarity_matrix(self, a, matrix):
        results:list = []
        for row in matrix:
            results.append(self.cosine_similarity(row, a))


In [19]:
def check(got, want, tol=1e-9):
    """Compare one result against its expected value."""
    if got is None:
        return "not implemented"
    try:
        return "OK" if abs(got - want) < tol else f"WRONG got {got!r} want {want!r}"
    except TypeError:
        return f"WRONG got {got!r} (expected a number)"


sol = Solution()

cases = [
    ([1, 2, 3], [2, 4, 6],  1.0),    # same direction, different length
    ([1, 0],    [0, 1],     0.0),    # orthogonal
    ([1, 0],    [-1, 0],   -1.0),    # opposite
    ([0, 0],    [1, 2],     0.0),    # zero vector -> 0.0, NOT nan
    ([0, 0],    [0, 0],     0.0),    # both zero
    ([1, 1],    [1, 0],     0.7071067811865476),   # 45 degrees
]

print("cosine_similarity")
for a, b, want in cases:
    print(f"  {str(a):<11} {str(b):<11} -> {check(sol.cosine_similarity(a, b), want, 1e-12)}")

print("\ncosine_similarity_unit")
for a, b, want in cases:
    print(f"  {str(a):<11} {str(b):<11} -> {check(sol.cosine_similarity_unit(a, b), want, 1e-12)}")

# no warning should be printed by the zero-vector cases above
got = sol.cosine_similarity([1, 1], [1, 0])
print("\nreturns exactly float:", type(got).__name__ == "float", f"(got {type(got).__name__})")

# the matrix version: expected [1.0, -1.0, 0.0]
M = np.array([[2.0, 4.0, 6.0],      # scaled copy of a  -> 1.0
              [-1.0, -2.0, -3.0],   # opposite          -> -1.0
              [0.0, 0.0, 0.0]])     # zero row          -> 0.0
print("matrix version       :", sol.cosine_similarity_matrix([1, 2, 3], M), " want [1. -1. 0.]")

cosine_similarity
  [1, 2, 3]   [2, 4, 6]   -> OK
  [1, 0]      [0, 1]      -> OK
  [1, 0]      [-1, 0]     -> OK
  [0, 0]      [1, 2]      -> OK
  [0, 0]      [0, 0]      -> OK
  [1, 1]      [1, 0]      -> OK

cosine_similarity_unit
  [1, 2, 3]   [2, 4, 6]   -> OK
  [1, 0]      [0, 1]      -> OK
  [1, 0]      [-1, 0]     -> OK
  [0, 0]      [1, 2]      -> OK
  [0, 0]      [0, 0]      -> OK
  [1, 1]      [1, 0]      -> OK

returns exactly float: False (got float64)
matrix version       : None  want [1. -1. 0.]


### After it passes: what cosine throws away

Three documents as word-count vectors. `long_` is the *same* document as `short`,
written out twenty times over. `other` is genuinely about something else.

Run it and compare the two metrics. Cosine is doing something L2 cannot, and the
reason is one word long.

In [ ]:
short = np.array([1.0, 2.0, 1.0])     # a short document
long_ = short * 20                    # the SAME document, 20x longer
other = np.array([4.0, 0.0, 1.0])     # a genuinely different document

print(f"cosine(short, long )  = {sol.cosine_similarity(short, long_)}")
print(f"cosine(short, other)  = {sol.cosine_similarity(short, other)}")
print()
print(f"L2    (short, long )  = {np.linalg.norm(short - long_):.4f}")
print(f"L2    (short, other)  = {np.linalg.norm(short - other):.4f}")
print()
print("which metric calls the first pair 'the same thing', and why?")